In [1]:
from kiteconnect import KiteConnect
import pandas as pd
import datetime as dt
import os
import numpy as np
from stocktrends import Renko

In [6]:
request_token=open("request_token.txt","r").read()
key_secrets=open("api_key.txt","r").read().split()
kite = KiteConnect(api_key=key_secrets[0])
data = kite.generate_session(request_token=request_token,api_secret=key_secrets[1])

In [7]:
# Get dump of all NSE instruments
instruments_dump=kite.instruments("NSE")
instruments_df=pd.DataFrame(instruments_dump)


In [8]:
def instrumentLookup(instruments_df,symbol):
    """Looks up instrument token for a given script from instrument dump"""
    try:
        return instruments_df[instruments_df.tradingsymbol==symbol].instrument_token.values[0]
    except:
        return -1


def fetchOHLC(ticker,interval,duration):
    """extracts historical data and outputs in the form of dataframe"""
    instrument = instrumentLookup(instruments_df,ticker)
    data = pd.DataFrame(kite.historical_data(instrument,dt.date.today()-dt.timedelta(duration), dt.date.today(),interval))
    data.set_index("date",inplace=True)
    return data

In [9]:
def hammer(ohlc_df):    
    """returns dataframe with hammer candle column"""
    df = ohlc_df.copy()
    df["hammer"] = (((df["high"] - df["low"])>3*(df["open"] - df["close"])) & \
                   ((df["close"] - df["low"])/(.001 + df["high"] - df["low"]) > 0.6) & \
                   ((df["open"] - df["low"])/(.001 + df["high"] - df["low"]) > 0.6)) & \
                   (abs(df["close"] - df["open"]) > 0.1* (df["high"] - df["low"]))
    return df

ohlc = fetchOHLC("ICICIBANK","5minute",30)
hammer_df = hammer(ohlc)

In [10]:
ohlc

,open,high,low,close,volume
date,,,,,
2025-03-13 09:15:00+05:30,1251.05,1254.95,1246.95,1248.15,593417
2025-03-13 09:20:00+05:30,1248.15,1248.20,1245.75,1246.80,134616
2025-03-13 09:25:00+05:30,1246.90,1248.50,1245.30,1247.90,90065
2025-03-13 09:30:00+05:30,1247.90,1248.55,1246.05,1246.95,103124
2025-03-13 09:35:00+05:30,1246.75,1250.65,1246.40,1250.65,72106
...,...,...,...,...,...
2025-04-11 15:05:00+05:30,1312.35,1313.80,1312.05,1312.30,489412
2025-04-11 15:10:00+05:30,1312.30,1312.35,1310.80,1311.35,472191
2025-04-11 15:15:00+05:30,1311.40,1311.40,1308.80,1309.60,536194


In [ ]:
hammer_df